# Checking ordinal

Proportional-odds (ordinal logit) model from statsmodels on simulated data: 5 ordered categories, temperature as the only covariate.

$$P(Y \le k \mid \text{temp}) = \sigma(c_k - \beta \cdot \text{temp}), \quad k = 1, \dots, 4$$

One shared slope $\beta$ and $K-1 = 4$ cutpoints. No intercept — the cutpoints play that role.

## Simulate data

In [1]:
import numpy as np
import pandas as pd
from statsmodels.miscmodels.ordinal_model import OrderedModel

rng = np.random.default_rng(0)
n = 500

# Covariate: room temperature (°C)
temp = rng.normal(22, 2, n)

# Latent model: y* = beta * temp + eps, eps ~ Logistic; 4 cutpoints -> 5 categories
beta_true = 1.2
cuts_true = np.array([24.0, 25.5, 27.0, 28.5])
y_star = beta_true * temp + rng.logistic(size=n)
y = np.searchsorted(cuts_true, y_star)  # 0..4

labels = ["very low", "low", "medium", "high", "very high"]
df = pd.DataFrame({
    "temp": temp,
    "level": pd.Categorical.from_codes(y, categories=labels, ordered=True),
})
print(df["level"].value_counts().sort_index())
df.head()

level
very low     110
low           78
medium       113
high          87
very high    112
Name: count, dtype: int64


,temp,level
0,22.251460,high
1,21.735790,medium
2,23.280845,medium
3,22.209800,very high
4,20.928661,low


## Fit

In [2]:
mod = OrderedModel(df["level"], df[["temp"]], distr="logit")
res = mod.fit(method="bfgs", disp=False)
print(res.summary())

                             OrderedModel Results                             
Dep. Variable:                  level   Log-Likelihood:                -575.04
Model:                   OrderedModel   AIC:                             1160.
Method:            Maximum Likelihood   BIC:                             1181.
Date:                Mon, 23 Nov 2026                                         
Time:                        07:57:27                                         
No. Observations:                 500                                         
Df Residuals:                     495                                         
Df Model:                           1                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
temp               1.2262      0.073     16.775      0.000       1.083       1.370
very low/low      24.6192      1.534    

In [3]:
# Summary shows first cut + log-increments; transform back to actual cutpoints
cuts_hat = mod.transform_threshold_params(res.params)[1:-1]
print("true cutpoints:     ", cuts_true)
print("estimated cutpoints:", cuts_hat.round(3))

true cutpoints:      [24.  25.5 27.  28.5]
estimated cutpoints: [24.619 25.977 27.665 29.183]


## Predicted category probabilities

In [4]:
grid = pd.DataFrame({"temp": [18, 20, 22, 24, 26]})
# np.asarray: predict returns its own 0..n index, which would reindex to NaN
probs = pd.DataFrame(np.asarray(res.predict(grid)), columns=labels, index=grid["temp"])
probs.round(3)

,very low,low,medium,high,very high
temp,,,,,
18,0.927,0.053,0.016,0.003,0.001
20,0.524,0.287,0.148,0.032,0.009
22,0.086,0.182,0.396,0.235,0.099
24,0.008,0.023,0.115,0.293,0.561
26,0.001,0.002,0.012,0.049,0.937
